In [ ]:
import pandas as pd

cd_74_associated_genes_table = pd.read_csv('CD74_Signature_Hs.csv', delimiter='\t')
# Display the first few rows
print(cd_74_associated_genes_table.head())
#createa a set of the genes in the table
cd_74_associated_genes_set=set(list(cd_74_associated_genes_table['Gene_ID']))

   Gene_ID
0    ALPK1
1    AZIN2
2  BHLHE40
3  BHLHE41
4     BLNK


In [ ]:
#read in the probe description table)
probe_table = pd.read_csv('illumnia_probe_list.txt', delimiter='\t')
# Display the first few rows
print(probe_table.head())

/tmp/ipykernel_4097111/2143752828.py:2: DtypeWarning: Columns (11,14,15,36) have mixed types. Specify dtype option on import or set low_memory=False.
  probe_table = pd.read_csv('illumnia_probe_list.txt', delimiter='\t')


           ID        Name  AddressA_ID  \
0  cg00035864  cg00035864     31729416   
1  cg00050873  cg00050873     32735311   
2  cg00061679  cg00061679     28780415   
3  cg00063477  cg00063477     16712347   
4  cg00121626  cg00121626     19779393   

                                    AlleleA_ProbeSeq  AddressB_ID  \
0  AAAACACTAACAATCTTATCCACATAAACCCTTAAATTTATCTCAA...          NaN   
1  ACAAAAAAACAACACACAACTATAATAATTTTTAAAATAAATAAAC...   31717405.0   
2  AAAACATTAAAAAACTAATTCACTACTATTTAATTACTTTATTTTC...          NaN   
3  TATTCTTCCACACAAAATACTAAACRTATATTTACAAAAATACTTC...          NaN   
4  AAAACTAATAAAAATAACTTACAAACCAAATACTATACCCTACAAC...          NaN   

                                    AlleleB_ProbeSeq Infinium_Design_Type  \
0                                                NaN                   II   
1  ACGAAAAAACAACGCACAACTATAATAATTTTTAAAATAAATAAAC...                    I   
2                                                NaN                   II   
3                       

In [ ]:
cd_74_associated_genes_table = pd.read_csv('CD74_Signature_Hs.csv', delimiter='\t')
#read in the probe description table
probe_table = pd.read_csv('illumnia_probe_list.txt', delimiter='\t')
#"We considered a CpG site to be located in a promoter region if it was annotated as TSS1500, TSS200, 5′UTR, or 1stExon."
#keel only the rows whichs  UCSC_RefGene_Group contains TSS1500, TSS200, 5′UTR, or 1stExon
promoter_only = probe_table[
    probe_table['UCSC_RefGene_Group']
    .astype(str)  # Convert to string
    .str.contains('TSS1500|TSS200|5\'UTR|1stExon', case=False, na=False)
]

filtered_df = promoter_only[
    promoter_only['UCSC_RefGene_Name'].apply(
        lambda x: any(gene in cd_74_associated_genes_set for gene in str(x).split(';'))
    )
]
#1308 CPG SITE LEFT (FIRST I FILTERED FOR PTOMOTER REGION THEN TO CG74 RELATION)
#THERE WOULD BE 2410 IF I WOULD ONLY FILTER FOR THE CG74 RELATION

In [ ]:
#WRITE THE FILTERED CPGS INTO A TXT      (these will be the models features later)
cd_74_associated_cpgs=list(filtered_df['ID'])
file_path='cd_74_associated_cpgs.txt'
with open(file_path, 'w') as file:
    for line in cd_74_associated_cpgs:
        file.write(line + "\n")

In [ ]:
#NOW GET THE cd_74_associated_cpgs -S GENE NAME INTO A DICTIONARY SO A CPG IS THE KEY AND THE GENE IS THE VALUE
#filtered_df[['UCSC_RefGene_Name','ID']]

cg_gene_dict = {key: None for key in cd_74_associated_cpgs}

gene_name_list=list(filtered_df['UCSC_RefGene_Name'])
cg_name_list=list(filtered_df['ID'])
#now search for every cg sites gene and place it in the value of the dict
i=0
for cg_id in cg_name_list:
    cg_gene_dict[cg_id]=gene_name_list[i]
    i=i+1
cg_gene_dict #print

{'cg03128268': 'RBM3',
 'cg12251508': 'RBM3',
 'cg12983165': 'RBM3',
 'cg16315447': 'RBM3',
 'cg20657691': 'RBM3',
 'cg24741068': 'RBM3;RBM3',
 'cg25656978': 'RBM3;RBM3',
 'cg26155374': 'RBM3',
 'cg27044041': 'RBM3',
 'cg27124847': 'RBM3',
 'cg27333993': 'RBM3',
 'cg00108454': 'C1QA',
 'cg00136477': 'C1QC;C1QC',
 'cg00215182': 'C1QB',
 'cg00396625': 'EPHX1;EPHX1',
 'cg00501904': 'EPHX1',
 'cg00524289': 'EPHX1',
 'cg00548060': 'NPL',
 'cg01444397': 'PER3',
 'cg01728495': 'C1QB',
 'cg02361903': 'CSF1;CSF1;CSF1;CSF1',
 'cg02814691': 'NPL',
 'cg03138928': 'EPHX1',
 'cg03337430': 'EPHX1;EPHX1',
 'cg03459809': 'EPHX1;EPHX1;EPHX1',
 'cg03941108': 'C1QB',
 'cg05005301': 'CSF1;CSF1;CSF1;CSF1',
 'cg05385434': 'EPHX1',
 'cg05605921': 'EPHX1',
 'cg05803631': 'PER3',
 'cg06487986': 'PER3',
 'cg07012832': 'C1QB',
 'cg07109332': 'C1QA',
 'cg07568430': 'CSF1;CSF1;CSF1;CSF1',
 'cg07813275': 'NPL',
 'cg08139855': 'CSF1;CSF1;CSF1;CSF1',
 'cg08463932': 'LAPTM5',
 'cg08578023': 'CTSS;CTSS',
 'cg08710757': 

In [ ]:
#THERE ARE PLACES WHERE MULTIPLE DIFFERENT GENE NAMES APPEAR FOR ONE CPG SITE, WHICH IS A PROBLEM
#THE cg_gene_dict.values() SHOULD BE SPLIT BY THE ; THEN COUNT THEM
print(cd_74_associated_genes_table.shape)
print(len(set(cg_gene_dict.values()))) 
#SINCE THE SHAPES DO NOT MATCH, I HAVE TO SPLIT THE VALUES OF THE DICT VALUES BY THE ;

(127, 1)
252


In [ ]:
#THIS MAKES SURE THAT THERE IS NO OTHER SEPARATOR IN THE GENE NAMES OTHER THAN ; SO OUR SEPARATION IS CORRECT
# Check for fields containing ' ', ':', ';', or '\t'
contains_special_chars = filtered_df['UCSC_RefGene_Name'].str.contains(r'[ :\t]')

# Display rows with special characters
rows_with_special_chars = filtered_df[contains_special_chars]
rows_with_special_chars['UCSC_RefGene_Name']

Series([], Name: UCSC_RefGene_Name, dtype: object)

In [42]:
#REMOVE DUPLICATIONS (CG.. = GENE1,GENE1,GENE2     ->       CG... = GENE1,GENE2)
#DELETE THE GENES WHICH ARE NOT CD74 RELATED, SINCE THERE CAN BE CG... = GENE1,GENE2   WHERE GENE2 IS NOT CD74 RELATED, SO WE DONT CARE
#    (THOUGH ONE OF THE GENES MUST BE CD74 RELATED SINCE WE HAVE ALREADY FILTERED THEM FOR THIS)

for key, value in cg_gene_dict.items():
    # Split the value string by ';'
    substrings = value.split(';')
    
    # Remove duplicates while preserving order
    #unique_substrings = list(dict.fromkeys(substrings))
    cd74_substrings = {s for s in substrings if s in cd_74_associated_genes_set}
    unique_substrings = list(set(cd74_substrings))

    # Join the unique substrings back into a single string
    cg_gene_dict[key] = ';'.join(unique_substrings)
    print(f"Key: {key}, {cg_gene_dict[key]}")

Key: cg03128268, RBM3
Key: cg12251508, RBM3
Key: cg12983165, RBM3
Key: cg16315447, RBM3
Key: cg20657691, RBM3
Key: cg24741068, RBM3
Key: cg25656978, RBM3
Key: cg26155374, RBM3
Key: cg27044041, RBM3
Key: cg27124847, RBM3
Key: cg27333993, RBM3
Key: cg00108454, C1QA
Key: cg00136477, C1QC
Key: cg00215182, C1QB
Key: cg00396625, EPHX1
Key: cg00501904, EPHX1
Key: cg00524289, EPHX1
Key: cg00548060, NPL
Key: cg01444397, PER3
Key: cg01728495, C1QB
Key: cg02361903, CSF1
Key: cg02814691, NPL
Key: cg03138928, EPHX1
Key: cg03337430, EPHX1
Key: cg03459809, EPHX1
Key: cg03941108, C1QB
Key: cg05005301, CSF1
Key: cg05385434, EPHX1
Key: cg05605921, EPHX1
Key: cg05803631, PER3
Key: cg06487986, PER3
Key: cg07012832, C1QB
Key: cg07109332, C1QA
Key: cg07568430, CSF1
Key: cg07813275, NPL
Key: cg08139855, CSF1
Key: cg08463932, LAPTM5
Key: cg08578023, CTSS
Key: cg08710757, C1QA
Key: cg08764927, PER3
Key: cg09766355, NPL
Key: cg10001720, LAPTM5
Key: cg10088685, C1QB
Key: cg10103528, C1QB
Key: cg10916651, C1QA
Ke

In [44]:
import json
# Save the dictionary to a file
with open('cg_gene_dict.json', 'w') as f:
    json.dump(cg_gene_dict, f)